# Beginner Real-World Data Science Projects

This notebook demonstrates a small end-to-end analysis using synthetic retail-style data: inspect data, clean it, explore trends, visualize results, and build a simple baseline regression model. The data is generated locally, so no download is needed.

The dataset is for practice only; it does not represent real customers or a real business.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

## 1. Generate a synthetic retail dataset

In [ ]:
n = 360
dates = pd.date_range("2024-01-01", periods=n, freq="D")
categories = np.array(["Books", "Home", "Sports", "Technology"])
category = rng.choice(categories, size=n, p=[0.25, 0.30, 0.20, 0.25])
price = rng.uniform(8, 180, size=n)
units = rng.integers(1, 9, size=n)
seasonal = 18 * np.sin(np.arange(n) * 2 * np.pi / 30)
category_effect = pd.Series(category).map({
    "Books": 0, "Home": 12, "Sports": 7, "Technology": 25
}).to_numpy()
sales = np.maximum(0, price * units * 0.25 + seasonal + category_effect + rng.normal(0, 18, n))

df = pd.DataFrame({
    "date": dates,
    "category": category,
    "price": price.round(2),
    "units": units,
    "sales": sales.round(2),
})
# Add a few missing values to practice data-quality checks.
missing_idx = rng.choice(df.index, size=8, replace=False)
df.loc[missing_idx, "price"] = np.nan
df.head()

## 2. Inspect and clean

In [ ]:
print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

df = df.drop_duplicates().copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
print("\nCleaned shape:", df.shape)

## 3. Explore summary statistics

In [ ]:
display(df.describe(include="all"))
print("Sales by category:")
display(df.groupby("category")["sales"].agg(["count", "mean", "sum"]).sort_values("sum", ascending=False))

## 4. Visualize distributions and trends

In [ ]:
df["sales"].plot(kind="hist", bins=25, title="Distribution of Daily Sales")
plt.xlabel("Sales")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

In [ ]:
category_sales = df.groupby("category")["sales"].sum().sort_values()
category_sales.plot(kind="barh", title="Total Sales by Category")
plt.xlabel("Total sales")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

In [ ]:
monthly_sales = df.groupby(df["date"].dt.to_period("M"))["sales"].sum()
monthly_sales.plot(kind="line", marker="o", title="Monthly Sales")
plt.xlabel("Month")
plt.ylabel("Total sales")
plt.tight_layout()
plt.show()

## 5. Build a simple baseline regression model

In [ ]:
# Example target: estimate sales from price, units, category, and calendar month.
model_df = df.dropna(subset=["sales", "date"]).copy()
model_df["month"] = model_df["date"].dt.month
features = ["price", "units", "category", "month"]
X = model_df[features]
y = model_df["sales"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

numeric_features = ["price", "units", "month"]
categorical_features = ["category"]
preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])

baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DummyRegressor(strategy="mean")),
])
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])

baseline.fit(X_train, y_train)
linear_model.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)
linear_pred = linear_model.predict(X_test)

metrics = pd.DataFrame([
    {"model": "DummyRegressor", "MAE": mean_absolute_error(y_test, baseline_pred),
     "RMSE": mean_squared_error(y_test, baseline_pred) ** 0.5,
     "R2": r2_score(y_test, baseline_pred)},
    {"model": "LinearRegression", "MAE": mean_absolute_error(y_test, linear_pred),
     "RMSE": mean_squared_error(y_test, linear_pred) ** 0.5,
     "R2": r2_score(y_test, linear_pred)},
])
metrics

## 6. Save project outputs

In [ ]:
summary = {
    "project": "beginner-retail-sales-analysis",
    "data_type": "synthetic",
    "rows": int(len(df)),
    "columns": list(df.columns),
    "missing_values": {k: int(v) for k, v in df.isna().sum().items()},
    "test_metrics": metrics.to_dict(orient="records"),
    "limitations": [
        "Synthetic data does not represent a real business.",
        "Associations in this dataset do not establish causation.",
        "A single random split is only a basic demonstration."
    ],
}
(ARTIFACT_DIR / "project_summary.json").write_text(
    __import__("json").dumps(summary, indent=2), encoding="utf-8"
)
metrics.to_csv(ARTIFACT_DIR / "model_metrics.csv", index=False)
print("Saved artifacts to:", ARTIFACT_DIR.resolve())

## 7. Write your findings

After running the notebook, write down:

1. Which category has the largest total sales in this generated sample?
2. What patterns do you see in the monthly sales chart?
3. Does linear regression improve on the dummy baseline under MAE and RMSE?
4. What would you need to validate before using a model for a real business?

Do not report numbers or findings until you have run the relevant cells. The dataset is synthetic and intended for learning, not business decisions.